In [ ]:
# Cellule 1 — Imports
import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.append('../src')

from riccati import OWParams, solve_riccati
from simulation import simulate_optimal_strategy, compute_internalization

plt.style.use('seaborn-v0_8-whitegrid')
print("Setup OK")

In [ ]:
# Cellule 2 — Sensibilité à l'autocorrélation (Table 1 de Nutz et al.)

scenarios = {
    'momentum': OWParams(theta=-1.0, sigma=0.1),
    'martingale': OWParams(theta=0.0, sigma=0.1),
    'reversal': OWParams(theta=1.0, sigma=0.1)
}

results = {}
for name, params in scenarios.items():
    res = simulate_optimal_strategy(params, n_paths=1000)
    intern = compute_internalization(res)
    results[name] = {
        'internalization': intern * 100,
        'mean_cost': res['mean_cost'],
        'res': res
    }
    print(f"{name:12s} | internalization: {intern*100:.1f}% | cost: {res['mean_cost']:.4f}")

In [ ]:
# Cellule 3 — Trajectoires moyennes (Figure 2 de Nutz et al.)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

colors = {'momentum': 'orange', 'martingale': 'blue', 'reversal': 'green'}
labels = {'momentum': 'momentum (θ=-1)', 
          'martingale': 'martingale (θ=0)', 
          'reversal': 'reversal (θ=1)'}

for name, res_dict in results.items():
    res = res_dict['res']
    t = res['t']
    color = colors[name]
    label = labels[name]

    # Trajectoires moyennes
    X_mean = np.mean(res['X'], axis=0)
    Y_mean = np.mean(res['Y'], axis=0)
    Z_mean = np.mean(res['Z'], axis=0)

    # Ecart-type
    X_std = np.std(res['X'], axis=0)
    Z_std = np.std(res['Z'], axis=0)

    axes[0].plot(t, Z_mean * 100, color=color, label=label)
    axes[0].fill_between(t, 
                          (Z_mean - X_std) * 100,
                          (Z_mean + X_std) * 100,
                          alpha=0.15, color=color)

    axes[1].plot(t, -X_mean * 100, color=color, label=label)
    axes[1].fill_between(t,
                          (-X_mean - X_std) * 100,
                          (-X_mean + X_std) * 100,
                          alpha=0.15, color=color)

    axes[2].plot(t, Y_mean * 10000, color=color, label=label)

# Formatting
axes[0].set_title('In-flow (ADV%)', fontsize=11)
axes[1].set_title('Inventory (ADV%)', fontsize=11)
axes[2].set_title('Impact state (bps)', fontsize=11)

for ax in axes:
    ax.set_xlabel('Time (days)')
    ax.legend(fontsize=8)

fig.suptitle('Optimal unwind strategy — sensitivity to flow autocorrelation\n(Nutz, Webster & Zhao 2025, Figure 3)', 
             fontsize=12)
plt.tight_layout()
plt.savefig('../figures/fig_nutz_trajectories.png', dpi=150)
plt.show()